#  Create MongoDB a

- autmated in script mongodb_ingest.py

In [1]:
import os
import pandas as pd
from datetime import datetime
import magic
import pydicom
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import csv
import re
from pymongo import MongoClient
import os
import pandas as pd
import pydicom
from pymongo import MongoClient

In [1]:
ANALYSIS_PATH='/home/bule/berner_daten_analyse/data'
df = pd.read_csv(os.path.join(ANALYSIS_PATH,"dicom_files_df.csv"),sep=',')

NameError: name 'pd' is not defined

In [ ]:
df.head()

In [ ]:
# MongoDB connection
client = MongoClient("mongodb://localhost:27017/")
db_name = "bern_db"
db = client[db_name]

# client.drop_database(db_name)

collection_name = "patients"
collection = db[collection_name]

# Check if the database exists
if collection.estimated_document_count() > 0:
    print(f"Database '{db_name}' already exists with collection '{collection_name}'.")
else:
    print(f"Creating database '{db_name}' with collection '{collection_name}'.")


In [10]:
def extract_dicom_metadata(file_path):
    """Extract all tags from a DICOM file dynamically, handling nested datasets."""
    try:
        ds = pydicom.dcmread(file_path, stop_before_pixels=True)  # Exclude pixel data for efficiency
        metadata = {}

        def parse_element(element):
            """Recursively parse a DICOM element."""
            if element.VR == "SQ":  # Sequence
                sequence_data = []
                for item in element.value:
                    item_metadata = {}
                    for sub_elem in item:
                        item_metadata[sub_elem.name] = parse_element(sub_elem)
                    sequence_data.append(item_metadata)
                return sequence_data
            elif isinstance(element.value, pydicom.dataset.Dataset):
                # Nested Dataset
                nested_metadata = {}
                for sub_elem in element.value:
                    nested_metadata[sub_elem.name] = parse_element(sub_elem)
                return nested_metadata
            else:
                # Simple element
                return str(element.value)  # Convert to string to ensure compatibility

        for elem in ds:
            metadata[elem.name] = parse_element(elem)

        return metadata
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return None


def populate_mongo_from_df_with_tags(df, mongo_db):
    """Populate MongoDB with hierarchical schema and dynamically extract metadata from DICOM files."""
    for _, row in df.iterrows():
        patient_id = row["Patient_ID"]
        study_id = row["Study_ID"]
        sequence_id = row["Sequence_ID"]
        full_path = row["Full_Path"]

        # Extract metadata from DICOM file
        metadata = extract_dicom_metadata(full_path)
        if metadata is None:
            continue

        # Upsert the hierarchical structure into MongoDB
        mongo_db["dicom_data"].update_one(
            {"_id": patient_id},  # Query to find the patient
            {
                "$set": {
                    f"{study_id}.{sequence_id}.{full_path}": {
                        "metadata": metadata
                    }
                }
            },
            upsert=True  # Create a new document if it doesn't exist
        )
        print(f"Inserted/Updated DICOM metadata for: {full_path}")
        
def populate_flattened_mongo(df, mongo_db):
    """Populate MongoDB with a flattened schema."""
    for _, row in df.iterrows():
        patient_id = row["Patient_ID"]
        study_id = row["Study_ID"]
        sequence_id = row["Sequence_ID"]
        full_path = row["Full_Path"]

        # Extract metadata from DICOM file
        metadata = extract_dicom_metadata(full_path)
        if metadata is None:
            continue

        # Insert the flattened document into MongoDB
        mongo_db["dicom_data"].insert_one({
            "patient_id": patient_id,
            "study_id": study_id,
            "sequence_id": sequence_id,
            "full_path": full_path,
            "metadata": metadata
        })
        print(f"Inserted metadata for: {full_path}")

In [ ]:
populate_flattened_mongo(df, db)